# BurstGPT 3: mixed-load training and causal adaptation stress tests

Run in order in a fresh runtime. Starting from the raw CSV, train fresh DQN checkpoints with episode progress disabled, select checkpoints on validation only, then compare FCFS, length-aware, static SLO-aware, frozen DQN and adapting DQN.

Train/validate at **1×/2×/4×** arrival load; evaluate at **1×/4×/8×** on short continuous windows. Multipliers compress arrival gaps, never SLOs/token lengths. GPU is optional for minibatch learning; simulation runs on CPU.

Do not reuse old clock-enabled checkpoints or old run folders. These evaluation groups have already been inspected: this is exploratory evaluation, not an untouched final test. Reserve fresh windows for confirmatory claims. Timing has batch=1 calibration with assumed, unvalidated batch contention—not real-GPU benchmarking.

**Quick mode (default):** run only the same 8× peak condition, all five policies, one seed. It does not sample requests, shorten the window, or change its queue/SLO setup. It is not a 1% sample or a guaranteed 100× speedup. Full training stays enabled unless you separately opt into shorter smoke-test training.

**Stability experiment v2:** new default uses elapsed-time bootstrapping consistently in training and adaptation, plus a conservative online-update budget. This is an untuned ablation, not a proven performance fix. Retrain under the chosen discount mode; don't switch discount semantics only after loading an old checkpoint.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
BRANCH = "burstgpt_3"  # Change to main after this branch is merged.
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    REPO = Path("/content/llm-request-triage")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "status", "--short"], check=True)
        print("Using the existing checkout. Fetch/switch explicitly if you need another revision.")
else:
    REPO = Path.cwd().resolve()
    if REPO.name == "notebooks":
        REPO = REPO.parent

required = ["src/llm_triage/rl/update_schedule.py", "experiments/adaptation_stress.py", "configs/burstgpt3_ranges.json",
            "scripts/prepare_burstgpt3.py", "src/llm_triage/workload/time_ranges.py"]
assert all((REPO / name).is_file() for name in required), "Checkout lacks BurstGPT 3 changes; use the pushed burstgpt_3 branch."
os.chdir(REPO)
assert sys.version_info >= (3, 11), "Use Python 3.11 or newer."
print("Repository:", REPO)
print("Git revision:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
extras = ".[dev,rl,analysis,gpu]" if IS_COLAB else ".[dev,rl,analysis]"
subprocess.run([sys.executable, "-m", "pip", "install", "-e", extras, "pandas>=2.0"], check=True)
for path in (REPO, REPO / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from importlib import invalidate_caches
invalidate_caches()

import base64
import csv
from datetime import datetime, timezone
import hashlib
import html
import importlib.metadata
import json
import shutil
import uuid
import zipfile

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display, Image, FileLink

from llm_triage.workload.burstgpt import sha256


## 1. Settings

Upload BurstGPT_3.csv to Drive once; no old ZIP is needed. Persist outputs on Drive to resume completed cases across disconnects. Reuse a run name only with identical settings/code.

Defaults limit training to 24 nonempty five-minute candidate windows and validation to 8, evenly spaced across each timeline, not outcome-selected. Every request inside each selected window is retained. 24 episodes is an initial budget, not proof of convergence. Load exposure cycles across 1×/2×/4×.

The 1-second TTFT target is a configurable analysis target, not a validated patience threshold or reward change. Completion SLO remains five seconds.

### Quick versus full

Set `RUN_MODE="quick"` to run the existing peak-heavy 8× case with all five policies and one selected seed. Switch to `"full"` for all configured windows/load factors/seeds. Window selection happens before filtering, so the peak timestamps and requests match full mode exactly.

`QUICK_TRAINING=False` keeps the configured training episodes, training loads, window limits and validation schedule unchanged. Optionally set it to `True` for a shorter **undertrained workflow smoke test**. This changes the checkpoint/training budget, not the evaluation traffic. Quick/short-training outputs use separate folders so they cannot be confused with full results. Neither quick mode promises a runtime nor reduces raw-data preparation cost.

### Stability controls (separate from quick/full conditions)

The conservative preset uses online LR=1e-5, warmup=2048 transitions, update interval=32 decisions, at least 1 simulated second and 32 newly observed outcomes since the previous update, and at most 100 updates per condition. These are initial development-ablation settings, not tuned recommendations. Outcomes include drops, not just service completions. No catch-up burst of updates is performed.

Elapsed-time mode uses `gamma ** (transition_elapsed_seconds / time_unit_seconds)` for the bootstrap target, in training AND adaptation. Zero-time decisions do not decay future value. Observed step rewards still use the unchanged reward model: individual events within a step are not separately time-discounted. The reference unit is simulated server time, **not original CSV time**.

For controlled comparisons, vary one axis at a time: decision+legacy, decision+conservative, elapsed_time+legacy, elapsed_time+conservative, each with a fresh run name/checkpoint for its discount mode. Changing both at once cannot attribute improvement to one change. The legacy schedule does not change which discount mode you selected.

For development ablations, explicitly set `EVALUATION_SPLIT="validation"` and use a new run name. This uses the burst-selected validation window and is not a held-out score. It does change which source window is evaluated. The default `"test"` keeps the exact original peak-heavy 8× window; no traffic changes are hidden behind the conservative preset.

The current target network syncs every 200 updates; the conservative 100-update budget keeps it anchored to its starting weights during that condition. This is another effect of the budget, not proof of improvement.


In [ ]:
CSV_MODE = "drive"  # @param ["drive", "upload"]
DRIVE_CSV_PATH = "/content/drive/MyDrive/BurstGPT_3.csv"  # @param {type:"string"}
LOCAL_CSV_PATH = str(REPO / "BurstGPT_3.csv")  # Used when running locally.
RUN_NAME = "load_stress_ablation_v3"  # @param {type:"string"}
RUN_MODE = "quick"  # @param ["quick", "full"]
QUICK_SEED = None  # None selects the first configured seed.
QUICK_TRAINING = False  # @param {type:"boolean"}
QUICK_TRAIN_EPISODES = 3  # Used only when QUICK_TRAINING=True. Undertrained smoke test.
SEEDS = [7]  # For final experiments consider [7, 17, 27].
EPISODES = 24  # @param {type:"integer"}
VALIDATE_EVERY = 4  # @param {type:"integer"}
RL_CANDIDATE_ORDER = "fcfs"  # @param ["fcfs", "length_aware"]
DROP_PENALTY = 0.2  # @param {type:"number"}
LOSS_SMOOTHING_UPDATES = 50  # Moving average over learner updates, not episodes.
# Compare separate runs: fcfs/.2, length_aware/.2, then winning ordering/.5 or 1.0.
# Use EVALUATION_SPLIT=validation for development; use a unique RUN_NAME per configuration.
EPISODE_SECONDS = 300  # @param {type:"integer"}
TRAIN_ARRIVAL_MULTIPLIERS = [1, 2, 4]
EVAL_ARRIVAL_MULTIPLIERS = [1, 4, 8]
TRAIN_WINDOW_LIMIT = 24
VALIDATION_WINDOW_LIMIT = 8
EVAL_WINDOW_SECONDS = 900
EVAL_WINDOW_METHODS = {"mixed_traffic": "typical", "peak_heavy": "burst"}
EVALUATION_SPLIT = "test"  # @param ["test", "validation"]
DEVELOPMENT_WINDOW_METHODS = {"validation": "burst"}
MANUAL_EVAL_WINDOWS = None
TTFT_TARGET_SECONDS = 1.0
TIMELINE_SOURCE_BIN_SECONDS = 60.0
MAX_REQUESTS_PER_WINDOW = 30000
PERSIST_RESULTS_TO_DRIVE = True
RUN_REPOSITORY_TESTS = False  # @param {type:"boolean"}
ADAPTATION_PRESET = "conservative"  # @param ["conservative", "legacy", "custom"]
DISCOUNT_MODE = "elapsed_time"  # @param ["elapsed_time", "decision"]
GAMMA = 0.99
DISCOUNT_TIME_UNIT_SECONDS = 1.0
DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]
from llm_triage.rl.devices import resolve_device
print("Requested learner:", DEVICE, "| Selected minibatch device:", resolve_device(DEVICE))
print("Simulator and single-request inference run on CPU in all modes.")

# Manual values apply to custom mode; presets below override them.
ONLINE_LEARNING_RATE = 0.0001
EPSILON = 0.0  # Only the adapting copy explores, if nonzero.
BATCH_SIZE = 64
WARMUP = 256
UPDATE_INTERVAL = 4
REPLAY_CAPACITY = 50000
MIN_UPDATE_SECONDS = 0.0
MIN_NEW_OUTCOMES = 0
MAX_ONLINE_UPDATES = None
from llm_triage.rl.update_schedule import adaptation_preset
preset = adaptation_preset(ADAPTATION_PRESET)
if preset:
    ONLINE_LEARNING_RATE = preset["learning_rate"]
    WARMUP = preset["warmup"]
    UPDATE_INTERVAL = preset["update_interval"]
    MIN_UPDATE_SECONDS = preset["min_update_seconds"]
    MIN_NEW_OUTCOMES = preset["min_new_outcomes"]
    MAX_ONLINE_UPDATES = preset["max_updates"]
print("Adaptation preset:", ADAPTATION_PRESET, preset)
print("Discount mode:", DISCOUNT_MODE, "gamma:", GAMMA, "time unit (simulated s):", DISCOUNT_TIME_UNIT_SECONDS)

assert RUN_MODE in {"quick", "full"}, "RUN_MODE must be quick or full."
assert not QUICK_TRAINING or RUN_MODE == "quick", "Short-training smoke tests require quick mode."
if QUICK_TRAINING:
    assert type(QUICK_TRAIN_EPISODES) is int and QUICK_TRAIN_EPISODES >= len(TRAIN_ARRIVAL_MULTIPLIERS), (
        "Use at least one training episode per configured load level."
    )
ACTIVE_EPISODES = QUICK_TRAIN_EPISODES if QUICK_TRAINING else EPISODES
ACTIVE_VALIDATE_EVERY = min(VALIDATE_EVERY, ACTIVE_EPISODES) if QUICK_TRAINING else VALIDATE_EVERY
run_name = RUN_NAME or datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ_") + uuid.uuid4().hex[:6]
if RUN_MODE == "quick":
    run_name += "_quick_short_training" if QUICK_TRAINING else "_quick"
assert Path(run_name).name == run_name and run_name not in {".", ".."}, "Use a simple folder name."
if IS_COLAB and PERSIST_RESULTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_ROOT = Path("/content/drive/MyDrive/burstgpt3_adaptation_stress")
else:
    RESULTS_ROOT = REPO / "results/runs/colab_burstgpt3_stress"
RUN_ROOT = RESULTS_ROOT / run_name
RUN_ROOT.mkdir(parents=True, exist_ok=True)
FIGURES = RUN_ROOT / "figures"
FIGURES.mkdir(exist_ok=True)
DATA_DIR = RUN_ROOT / "data"
ANALYSIS_DIR = RUN_ROOT / "traffic_analysis"
TRAIN_OUTPUT = RUN_ROOT / "training_run"
COMMAND_LOG = RUN_ROOT / "notebook_commands.log"
RANGE_CONFIG_PATH = RUN_ROOT / "ranges.json"
TRAIN_CONFIG_PATH = RUN_ROOT / "training_config.json"
generated_figures = set()
print("Run folder:", RUN_ROOT)

def run_command(args):
    command = [str(value) for value in args]
    print("Running:", " ".join(command), flush=True)
    with COMMAND_LOG.open("a", encoding="utf-8") as log:
        log.write("\n$ " + " ".join(command) + "\n")
        with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

def save_figure(fig, name):
    path = FIGURES / (name + ".png")
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(fig)
    generated_figures.add(path.name)
    display(Image(filename=str(path)))
    return path

def save_config(path, value):
    if path.exists():
        assert json.loads(path.read_text()) == value, "Existing run config differs. Use a new RUN_NAME."
    else:
        path.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")


In [ ]:
if IS_COLAB:
    SOURCE_CSV = REPO / "data" / "raw" / "BurstGPT_3.csv"
    SOURCE_CSV.parent.mkdir(parents=True, exist_ok=True)
    if not SOURCE_CSV.exists():
        if CSV_MODE == "drive":
            from google.colab import drive
            drive.mount("/content/drive")
            drive_source = Path(DRIVE_CSV_PATH)
            assert drive_source.is_file(), "Set DRIVE_CSV_PATH to your BurstGPT_3.csv."
            shutil.copyfile(drive_source, SOURCE_CSV)
        elif CSV_MODE == "upload":
            from google.colab import files
            uploaded = files.upload()
            csv_files = [name for name in uploaded if Path(name).suffix.lower() == ".csv"]
            assert len(csv_files) == 1, "Upload exactly one BurstGPT CSV."
            SOURCE_CSV.write_bytes(uploaded[csv_files[0]])
            del uploaded
        else:
            raise ValueError("CSV_MODE must be drive or upload")
else:
    SOURCE_CSV = Path(LOCAL_CSV_PATH).expanduser().resolve()

assert SOURCE_CSV.is_file(), "BurstGPT_3.csv was not found."
print("CSV:", SOURCE_CSV, "Bytes:", SOURCE_CSV.stat().st_size)
if RUN_REPOSITORY_TESTS:
    run_command([sys.executable, "-m", "pytest", "-q"])


## 2. Prepare explicit chronological splits

Keep continuous traffic and idle gaps inside each episode. Explicit ranges override the analyzer's older automatic 60/20/20 labels. Training and checkpoint selection never use test outcomes.


In [ ]:
range_config = json.loads((REPO / "configs/burstgpt3_ranges.json").read_text())
range_config["episode_seconds"] = EPISODE_SECONDS
# Optional: edit range_config["ranges"] here before preparation.
assert sha256(SOURCE_CSV) == range_config["source_sha256"], "CSV does not match the selected BurstGPT 3 source."
save_config(RANGE_CONFIG_PATH, range_config)
display(pd.DataFrame(range_config["ranges"]))

if not (ANALYSIS_DIR / "summary.json").exists():
    run_command([sys.executable, "scripts/analyze_burstgpt_traffic.py", "--source", SOURCE_CSV,
                 "--output-dir", ANALYSIS_DIR, "--bin-seconds", 300])
analysis = json.loads((ANALYSIS_DIR / "summary.json").read_text())
assert analysis["source"]["sha256"] == range_config["source_sha256"]

if not (DATA_DIR / "processed" / "manifest.json").exists():
    run_command([sys.executable, "scripts/prepare_burstgpt3.py", "--source", SOURCE_CSV,
                 "--analysis-dir", ANALYSIS_DIR, "--ranges-config", RANGE_CONFIG_PATH, "--data-dir", DATA_DIR])
manifest = json.loads((DATA_DIR / "processed" / "manifest.json").read_text())
assert manifest["range_config"] == range_config, "Prepared ranges differ. Use a fresh run."
assert manifest["source_sha256"] == sha256(SOURCE_CSV)
assert manifest["selected_sha256"] == sha256(DATA_DIR / "processed" / "burstgpt_windows.csv")

episode_table = pd.DataFrame(manifest["windows"] + manifest.get("empty_windows", []))
episode_table["duration_s"] = episode_table["end_timestamp_s"] - episode_table["start_timestamp_s"]
episode_table["nonempty"] = episode_table["requests"] > 0
dataset_summary = episode_table.groupby(["split", "group"], sort=False).agg(
    requests=("requests", "sum"), episodes=("window", "size"),
    nonempty_episodes=("nonempty", "sum"), duration_s=("duration_s", "sum")).reset_index()
dataset_summary["no_record_episode_fraction"] = 1 - dataset_summary["nonempty_episodes"] / dataset_summary["episodes"]
dataset_summary["mean_rps_including_gaps"] = dataset_summary["requests"] / dataset_summary["duration_s"]
display(dataset_summary)
dataset_summary.to_csv(RUN_ROOT / "dataset_summary.csv", index=False)

traffic = pd.read_csv(ANALYSIS_DIR / "time_bins.csv")
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(traffic["start_timestamp_s"], traffic["mean_rps"], linewidth=0.6, color="0.5")
colors = {"train": "#2166ac", "validation": "#4daf4a", "test": "#d95f02"}
for item in range_config["ranges"]:
    ax.axvspan(item["start_timestamp_s"], item["end_timestamp_s"], alpha=0.18,
               color=colors[item["split"]], label=item["group"])
ax.ticklabel_format(axis="x", style="plain", useOffset=False)
ax.set(xlabel="Original CSV timestamp (seconds)", ylabel="Five-minute requests/s",
       title="Selected traffic ranges — natural traffic mix retained")
ax.legend()
ax.grid(alpha=0.2)
save_figure(fig, "selected_traffic_ranges")


### Select and lock short evaluation windows before running policies

Select a complete 15-minute block per group using traffic only: nearest median-count nonempty block for mixed traffic, highest five-minute request rate for peak-heavy traffic. No policy scores or empty prefixes. Labels are relative to these groups; 8× does not guarantee overload everywhere.

Each window/load condition is independent, starts empty and has **no internal five-minute queue resets**. Each adapting copy starts fresh from its source checkpoint; no learning carries between independent conditions. This measures adaptation *within* a window, not persistent learning across the old multi-day stream.

Optional MANUAL_EVAL_WINDOWS must contain name, group, split="test", start_timestamp_s and end_timestamp_s. Each must lie inside the corresponding test range. Reusing a run name locks the chosen windows.


In [ ]:
from experiments.adaptation_stress import (
    select_evaluation_windows, validate_evaluation_windows, select_run_conditions,
)

# Select the complete original windows FIRST. Quick mode only filters conditions.
all_evaluation_windows = MANUAL_EVAL_WINDOWS if MANUAL_EVAL_WINDOWS is not None else select_evaluation_windows(
    manifest, seconds=EVAL_WINDOW_SECONDS, split=EVALUATION_SPLIT,
    methods=DEVELOPMENT_WINDOW_METHODS if EVALUATION_SPLIT == "validation" else EVAL_WINDOW_METHODS)
validate_evaluation_windows(manifest, all_evaluation_windows, split=EVALUATION_SPLIT)
evaluation_windows, ACTIVE_EVAL_MULTIPLIERS, ACTIVE_SEEDS = select_run_conditions(
    all_evaluation_windows, EVAL_ARRIVAL_MULTIPLIERS, SEEDS, mode=RUN_MODE, quick_seed=QUICK_SEED,
    quick_group="validation" if EVALUATION_SPLIT == "validation" else "peak_heavy")
execution_plan = dict(
    mode=RUN_MODE, short_training=QUICK_TRAINING, evaluation_split=EVALUATION_SPLIT,
    adaptation_preset=ADAPTATION_PRESET, discount_mode=DISCOUNT_MODE, gamma=GAMMA,
    discount_time_unit_s=DISCOUNT_TIME_UNIT_SECONDS,
    interpretation="UNDERTRAINED SMOKE TEST" if QUICK_TRAINING else
                   ("Quick single-condition comparison" if RUN_MODE == "quick" else "Full configured comparison"),
    original_windows=all_evaluation_windows, selected_windows=evaluation_windows,
    original_multipliers=EVAL_ARRIVAL_MULTIPLIERS, selected_multipliers=ACTIVE_EVAL_MULTIPLIERS,
    seeds=ACTIVE_SEEDS, training_episodes=ACTIVE_EPISODES, validate_every=ACTIVE_VALIDATE_EVERY,
    request_sampling=False, window_shortening=False,
)
save_config(RUN_ROOT / "execution_plan.json", execution_plan)
save_config(RUN_ROOT / "evaluation_windows.json", evaluation_windows)
display(pd.DataFrame(evaluation_windows))
print(execution_plan["interpretation"])
print("Training episodes per seed:", ACTIVE_EPISODES)
print("Expected evaluation cases:", len(evaluation_windows) * len(ACTIVE_EVAL_MULTIPLIERS) * (3 + 2 * len(ACTIVE_SEEDS)))


## 3. Train new clock-disabled models

Keep the 15-feature shape, but make episode_progress constantly zero during training, validation and replay. Fresh retraining is required. Hardware, reward, SLO and network defaults stay unchanged. Checkpoint selection averages validation SLO attainment across selected window/load pairs; 8× evaluation outcomes are never used.

Interrupted training uses existing pipeline resume: completed training is reused; an incomplete training attempt restarts. Evaluation below supports finer per-policy caching.


In [ ]:
train_config = json.loads((REPO / "configs/burstgpt3_initial.json").read_text())
train_config.update(seeds=ACTIVE_SEEDS, episodes=ACTIVE_EPISODES, validate_every=ACTIVE_VALIDATE_EVERY,
                    target_rps=[None], arrival_multipliers=TRAIN_ARRIVAL_MULTIPLIERS,
                    observation={"include_episode_progress": False}, rl_candidate_order=RL_CANDIDATE_ORDER,
                    cohort_window_limit={"train": TRAIN_WINDOW_LIMIT, "validation": VALIDATION_WINDOW_LIMIT})
train_config.setdefault("agent", {})["device"] = DEVICE
train_config.setdefault("reward", {})["drop"] = DROP_PENALTY
train_config["agent"].update(gamma=GAMMA, discount_mode=DISCOUNT_MODE,
                             discount_time_unit_s=DISCOUNT_TIME_UNIT_SECONDS)
save_config(TRAIN_CONFIG_PATH, train_config)
display(train_config)
command = [sys.executable, "-m", "experiments.run_pipeline", "--config", TRAIN_CONFIG_PATH,
           "--data-dir", DATA_DIR, "--output", TRAIN_OUTPUT]
if (TRAIN_OUTPUT / "pipeline.json").exists():
    command.append("--resume")
run_command(command)
pipeline_state = json.loads((TRAIN_OUTPUT / "pipeline.json").read_text())
assert pipeline_state["status"] == "completed"
TRAINING_DIR = TRAIN_OUTPUT / "training"
training_record = json.loads((TRAINING_DIR / "run.json").read_text())
assert training_record["test_evaluated"] is False
checkpoint_hashes = {f"dqn_seed{item['seed']}.npz": item["sha256"] for item in training_record["checkpoints"]}
for filename, expected in checkpoint_hashes.items():
    assert sha256(TRAINING_DIR / filename) == expected
display(pd.DataFrame(training_record["checkpoints"]))


In [ ]:
history = pd.read_csv(TRAINING_DIR / "training.csv")
selection = pd.read_csv(TRAINING_DIR / "checkpoints.csv")
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for seed in history["seed"].unique():
    h = history[history["seed"] == seed]
    s = selection[selection["seed"] == seed]
    axes[0, 0].plot(s["episode"], s["validation_attainment"], marker="o", label=f"seed {seed}")
    axes[0, 1].plot(h["episode"], h["train_attainment"], label=f"seed {seed}")
    axes[1, 0].plot(h["episode"], h["reward"], label=f"seed {seed}")
    updates = pd.read_csv(TRAINING_DIR / f"training_updates_seed{seed}.csv")
    axes[1, 1].plot(updates["update"], updates["loss"], alpha=.15)
    axes[1, 1].plot(updates["update"], updates["loss"].rolling(LOSS_SMOOTHING_UPDATES, min_periods=1).mean(), label=f"seed {seed}")
for item in training_record["checkpoints"]:
    axes[0, 0].scatter(item["episode"], item["validation_attainment"], marker="*", s=150, color="black")
axes[0, 0].set_title("Validation attainment; stars mark selected checkpoints")
axes[0, 1].set_title("Training attainment (exploration enabled)")
axes[1, 0].set_title("Episode return (affected by request count)")
axes[1, 1].set_title("Update loss and moving average")
for ax in axes.flat:
    ax.set_xlabel("Training episode")
    ax.grid(alpha=0.2)
    ax.legend()
axes[1, 1].set_xlabel("Learner update")
for ax in axes[0]:
    ax.yaxis.set_major_formatter(PercentFormatter(1))
save_figure(fig, "learning_curves")
coverage = history.groupby("seed").agg(sampled_episodes=("episode", "size"), unique_windows=("window", "nunique"))
display(coverage)
display(selection[["seed", "episode", "updates", "validation_attainment", "validation_drop_rate"]])


### Validation comparison

These scores cover selected validation window/load pairs, not the complete validation range. Inspect arrival_multiplier in validation.csv and training.csv for exposure at each load.


In [ ]:
validation_runs = pd.read_csv(TRAIN_OUTPUT / "validation.csv")
validation_runs["dropped_estimate"] = validation_runs["drop_rate"] * validation_runs["arrivals"]
validation_summary = validation_runs.groupby("policy", sort=False).agg(
    requests=("arrivals", "sum"), slo_met=("slo_met", "sum"),
    dropped=("dropped_estimate", "sum"), mean_episode_attainment=("slo_attainment", "mean")).reset_index()
validation_summary["overall_slo_attainment"] = validation_summary["slo_met"] / validation_summary["requests"]
validation_summary["overall_drop_rate"] = validation_summary["dropped"] / validation_summary["requests"]
display(validation_summary)
validation_summary.to_csv(RUN_ROOT / "validation_weighted_summary.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, metric, title in zip(axes, ["overall_slo_attainment", "overall_drop_rate"],
                             ["Validation: request-weighted SLO attainment", "Validation: request-weighted drop rate"]):
    ax.bar(validation_summary["policy"], validation_summary[metric])
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.tick_params(axis="x", rotation=25)
    ax.set_title(title)
    ax.set_ylim(bottom=0)
    ax.grid(axis="y", alpha=0.2)
save_figure(fig, "validation_comparison")


## 4. Paired, past-feedback-only replay

All five policies receive identical requests and scaled arrivals per condition. Each DQN seed has paired frozen/adapting copies. Deterministic baselines run once per condition, not once per seed.

Frozen weights never change. Adaptation learns only after returned transitions/rewards/outcomes, starting with a fresh optimizer/buffer. Both copies are greedy by default; nonzero EPSILON enables exploration only for adaptation and changes the experiment.

Queues persist inside each window and drain fully. Learner wall-time is recorded separately, not charged as simulated GPU service. Completed policy cases are checksum-verified and skipped on rerun. Interrupted cases restart from their original checkpoint, never resume with missing queue/optimizer state.

The online budget uses elapsed *simulation* time and delivered outcomes; it never updates from future completions. Warmup, decision interval, time gate, feedback gate, and total budget must all pass. A short condition can legitimately end with no updates; inspect the learning summary. Frozen never updates, regardless of preset.


In [ ]:
from experiments.adaptation_stress import run as run_stress

ONLINE_OUTPUT = RUN_ROOT / "stress_comparison"
online_settings = dict(
    multipliers=ACTIVE_EVAL_MULTIPLIERS, seeds=ACTIVE_SEEDS, ttft_target_s=TTFT_TARGET_SECONDS,
    timeline_source_bin_s=TIMELINE_SOURCE_BIN_SECONDS, max_requests=MAX_REQUESTS_PER_WINDOW,
    epsilon=EPSILON, learning_rate=ONLINE_LEARNING_RATE, batch_size=BATCH_SIZE,
    warmup=WARMUP, update_interval=UPDATE_INTERVAL, capacity=REPLAY_CAPACITY, device=DEVICE,
    min_update_seconds=MIN_UPDATE_SECONDS, min_new_outcomes=MIN_NEW_OUTCOMES, max_updates=MAX_ONLINE_UPDATES,
    evaluation_split=EVALUATION_SPLIT,
)
display(pd.DataFrame([{key: str(value) for key, value in online_settings.items()}]))
run_stress(TRAINING_DIR, DATA_DIR, ONLINE_OUTPUT, evaluation_windows, **online_settings)
summary = pd.read_csv(ONLINE_OUTPUT / "summary.csv")
timeline = pd.read_csv(ONLINE_OUTPUT / "timeline.csv")
learning = pd.read_csv(ONLINE_OUTPUT / "learning_summary.csv")
length_groups = pd.read_csv(ONLINE_OUTPUT / "length_groups.csv")
for frame in (summary, timeline):
    frame["policy_label"] = frame.apply(
        lambda row: row.policy if pd.isna(row.seed) else f"{row.policy}_seed{int(row.seed)}", axis=1)
display(summary[["group", "multiplier", "policy_label", "arrivals", "slo_attainment", "drop_rate",
                 "window_goodput_rps", "p50_ttft_s", "p95_ttft_s", "p99_ttft_s",
                 "ttft_target_attainment", "first_token_coverage", "p95_completion_latency_s"]])
display(learning)


## 5. Unified scorecard

TTFT is arrival-to-first-token latency. Percentiles exclude no-token requests: also show first-token coverage and TTFT-target attainment over **all arrivals**. The TTFT target is separate from the completion SLO.

Completion-SLO attainment includes drops. Window goodput counts on-time completions inside the arrival window; cohort SLO includes drain. Wasted GPU fraction includes the full drained replay. Latency percentiles require an observed latency, so interpret them with drop/coverage rates.

Queue/batch/KV are duration-weighted iteration snapshots, not real GPU counters. Isolated service time and output-length groups use truth **offline after simulation**, never as policy input. Length buckets: input ≤512/≤2048/>2048 tokens; output ≤32/≤128/>128 tokens.

Timeline scores use eventual arrival-cohort outcomes for reporting only, not earlier decisions. Their cohort-local goodput is not instantaneous server-wide completion rate.


In [ ]:
metrics = [
    ("slo_attainment", "Completion-SLO attainment", True),
    ("window_goodput_rps", "Window on-time goodput (req/s)", False),
    ("drop_rate", "Drop rate", True),
    ("p50_ttft_s", "p50 TTFT (seconds)", False),
    ("p95_ttft_s", "p95 TTFT (seconds)", False),
    ("p99_ttft_s", "p99 TTFT (seconds)", False),
    ("ttft_target_attainment", "TTFT-target attainment / all arrivals", True),
    ("first_token_coverage", "First-token coverage", True),
    ("p95_completion_latency_s", "p95 completion latency (seconds)", False),
    ("p95_queue_wait_s", "p95 queue wait (served, seconds)", False),
    ("full_replay_wasted_gpu_fraction", "Wasted GPU fraction (incl. drain)", True),
    ("busy_fraction_in_window", "Busy fraction (arrival window)", True),
]
for window in evaluation_windows:
    frame = summary[summary.window == window["name"]]
    fig, axes = plt.subplots(4, 3, figsize=(16, 15), constrained_layout=True)
    for ax, (metric, title, fraction) in zip(axes.flat, metrics):
        for name, values in frame.groupby("policy_label"):
            values = values.sort_values("multiplier")
            ax.plot(values.multiplier, values[metric], "o-", label=name)
        ax.set(title=title, xlabel="Arrival multiplier")
        ax.set_xticks(sorted(frame.multiplier.unique()))
        if fraction:
            ax.yaxis.set_major_formatter(PercentFormatter(1))
            lower, upper = ax.get_ylim()
            ax.set_ylim(max(0, lower), min(1, upper))
        ax.grid(alpha=.2)
    axes[0, 0].legend(fontsize=8)
    fig.suptitle(window["name"])
    save_figure(fig, window["name"] + "_load_curves")

keys = ["group", "window", "multiplier", "seed"]
paired = summary[summary.policy == "adapting"].merge(
    summary[summary.policy == "frozen"], on=keys, suffixes=("_adapting", "_frozen"), validate="one_to_one")
delta = paired[keys].copy()
delta["slo_attainment_delta"] = paired.slo_attainment_adapting - paired.slo_attainment_frozen
delta["p95_ttft_delta_s"] = paired.p95_ttft_s_adapting - paired.p95_ttft_s_frozen
delta.to_csv(RUN_ROOT / "attainment_delta.csv", index=False)
display(delta)


In [ ]:
# Display individual seed curves; do not average percentiles across seeds.
for window in evaluation_windows:
    for multiplier in ACTIVE_EVAL_MULTIPLIERS:
        frame = timeline[(timeline.window == window["name"]) & (timeline.multiplier == multiplier)]
        fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
        for label, values in frame.groupby("policy_label"):
            values = values.sort_values("source_start_s")
            for ax, metric in zip(axes, ["slo_attainment", "p95_ttft_s", "mean_queue"]):
                ax.plot(values.source_start_s, values[metric], label=label)
        for ax, title in zip(axes, ["Arrival-cohort SLO attainment", "Arrival-cohort p95 TTFT (s)", "Mean queue"]):
            ax.set(title=title, xlabel="Source timestamp (seconds)")
            ax.ticklabel_format(axis="x", style="plain", useOffset=False)
            ax.grid(alpha=.2)
        axes[0].yaxis.set_major_formatter(PercentFormatter(1))
        axes[0].set_ylim(0, 1.05)
        axes[1].axhline(TTFT_TARGET_SECONDS, linestyle="--", color="gray", label="TTFT target")
        axes[0].legend(fontsize=7)
        fig.suptitle(f'{window["name"]} / {multiplier:g}x')
        save_figure(fig, f'{window["name"]}_load_{multiplier:g}_timeline')
display(length_groups[["group", "multiplier", "policy", "seed", "dimension", "length_group",
                       "arrivals", "slo_attainment", "drop_rate", "p95_ttft_s",
                       "first_token_coverage", "ttft_target_attainment"]])


## Learning diagnostics and objective

Per-transition logs record pre-update Q-values/masks, action, simulated elapsed time, observed reward components, other-request outcome counts, queues, and update counts. Feedback can belong to earlier requests: these are global step rewards, not proof that the latest action caused all outcomes.

Plots include decision frequencies, signed reward components, cumulative updates and Q gaps. Decision-level DEFER/DROP shares are not request drop rates. The reward totals are actual undiscounted observed rewards, not discounted training return or SLO attainment. Primary checkpoint selection/evaluation uses completion-SLO attainment, while the unchanged shaped reward has additional trade-offs.


In [ ]:
# Streaming CSV logs contain only information available at each returned step.
# Read one policy/condition at a time; aggregate by simulated time for plotting.
component_cols = ["reward_success", "reward_late", "reward_wasted_service", "reward_drop", "reward_defer"]
for window in evaluation_windows:
    for multiplier in ACTIVE_EVAL_MULTIPLIERS:
        for seed in ACTIVE_SEEDS:
            for policy in ("frozen", "adapting"):
                folder = ONLINE_OUTPUT / window["name"] / f"load_{multiplier:g}" / f"{policy}_seed{seed}"
                log = pd.read_csv(folder / f"{policy}_transitions.csv")
                if log.empty:
                    print("No runnable decisions:", folder)
                    continue
                width = TIMELINE_SOURCE_BIN_SECONDS / multiplier
                log["time_bin"] = (log.sim_after_s // width).astype(int)
                groups = log.groupby("time_bin", sort=True)
                times = groups.sim_after_s.max()
                counts = log.groupby(["time_bin", "action"]).size().unstack(fill_value=0)
                fractions = counts.div(counts.sum(axis=1), axis=0)
                fig, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)
                for action in ("ADMIT", "DEFER", "DROP"):
                    if action in fractions:
                        axes[0, 0].plot(times.reindex(fractions.index), fractions[action], "o-", label=action)
                axes[0, 0].set(title="Decision frequencies (not request drop rates)", ylabel="Fraction of decisions")
                axes[0, 0].yaxis.set_major_formatter(PercentFormatter(1))
                axes[0, 0].set_ylim(0, 1.05)
                components = groups[component_cols].sum()
                for column in component_cols:
                    axes[0, 1].plot(times, components[column], "o-", label=column.replace("reward_", ""))
                axes[0, 1].set(title="Observed reward components / time bin", ylabel="Signed reward")
                axes[1, 0].step(times, groups.updates.max(), where="post", marker="o", label="updates")
                if policy == "adapting" and MAX_ONLINE_UPDATES is not None:
                    axes[1, 0].axhline(MAX_ONLINE_UPDATES, linestyle="--", color="gray", label="budget")
                axes[1, 0].set(title="Learner updates vs simulated time", ylabel="Cumulative updates")
                q_gap = groups.q_admit.mean() - groups.q_drop.mean()
                axes[1, 1].plot(times, q_gap, "o-", label="Q(admit) - Q(drop)")
                axes[1, 1].axhline(0, color="gray", linestyle="--")
                axes[1, 1].set(title="Mean pre-update Q gap (diagnostic only)", ylabel="Q-value difference")
                for ax in axes.flat:
                    ax.set_xlabel("Simulated seconds")
                    ax.grid(alpha=.2)
                    ax.legend(fontsize=8)
                fig.suptitle(f'{window["name"]} / {multiplier:g}x / {policy} seed {seed}')
                save_figure(fig, f'{window["name"]}_load_{multiplier:g}_{policy}_seed{seed}_learning_diagnostics')
                del log

reward_columns = [c for c in learning if c.startswith("reward_")]
display(learning[["policy", "seed", "updates", "zero_time_steps", "cumulative_reward", *reward_columns]])
print("Primary evaluation metric remains completion-SLO attainment.")
print("The shaped reward also trades off drops, lateness, deferral and wasted GPU time; it is not the same objective.")


## 6. Interpret and share

Compare RL against length-aware and static SLO-aware, not only FCFS. Adaptation is not guaranteed to help; a short window may have too few observations. Review updates, action counts, coverage and request-length outcomes alongside latency.

Keep reward and network size unchanged initially. For final exploratory comparisons use multiple training seeds, show each seed, and do not count repeated deterministic baselines as independent evidence. Additional predeclared windows are needed to assess traffic variability.

The original multi-day persistent-adaptation protocol remains available through experiments.online_adaptation. Its carryover/reset semantics differ; results are not directly interchangeable.


In [ ]:
provenance = dict(
    experiment="Independent-window five-policy load stress (exploratory reused evaluation groups)",
    git_revision=subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip(),
    training_config=train_config, settings=online_settings, evaluation_windows=evaluation_windows,
    execution_plan=execution_plan,
    source_sha256=sha256(SOURCE_CSV), source_checkpoints=checkpoint_hashes,
    protocol=json.loads((ONLINE_OUTPUT / "run.json").read_text()),
    notebook_sha256=sha256(REPO / "notebooks/colab_burstgpt3_online_adaptation.ipynb"),
)
save_config(RUN_ROOT / "online_provenance.json", provenance)
parts = [
    "<!doctype html><html><head><meta charset='utf-8'><title>Adaptation load stress</title>",
    "<style>body{font:15px system-ui;max-width:1500px;margin:30px;padding:20px}img{max-width:100%}"
    "table{display:block;overflow-x:auto;border-collapse:collapse;font-size:12px}"
    "td,th{border:1px solid #ddd;padding:5px}pre{white-space:pre-wrap}</style></head><body>",
    "<h1>Five-policy adaptation load-stress experiment</h1>",
    "<h2>" + html.escape(execution_plan["interpretation"]) + "</h2>",
    "<p>Quick mode filters complete conditions, not requests or timestamps. "
    "Short-training results, if enabled, are undertrained smoke tests—not equivalent policy-quality evidence.</p>",
    "<p>Exploratory reused evaluation groups, not untouched final tests. Independent windows/load conditions, "
    "each starting empty and from identical source weights. Episode clock disabled in all RL phases. "
    "TTFT percentiles exclude no-token requests; target attainment includes all arrivals. "
    "Completion SLO includes drops. Window goodput excludes drain; GPU waste includes drain. "
    "Queue/batch/KV are approximate snapshots. Learner wall-time is not charged as simulated GPU service. "
    "Batch contention is assumed, not calibrated. True output lengths are offline diagnostics only.</p>",
    "<h2>Selected windows</h2>", pd.DataFrame(evaluation_windows).to_html(index=False),
    "<h2>Training and validation</h2>", pd.DataFrame(training_record["checkpoints"]).to_html(index=False),
    selection.to_html(index=False),
    validation_summary.to_html(index=False),
    "<h2>All policies and loads</h2>", summary.to_html(index=False),
    "<h2>Paired adapting minus frozen</h2>", delta.to_html(index=False),
    "<h2>Learning and execution</h2>", learning.to_html(index=False),
    "<p>Primary evaluation is completion-SLO attainment. Shaped reward also penalizes late work, drops, "
    "deferral and GPU waste, so higher reward is not necessarily higher attainment. Reward weights and RL ordering are recorded in training_config. "
    "Diagnostics show observed, undiscounted reward totals. Q gaps are not counterfactual performance. "
    "Update timing is simulated time; outcomes include drops. Bootstrap discount semantics are recorded below.</p>",
    "<h2>Length-group diagnostics</h2>", length_groups.to_html(index=False),
    "<p>Timeline uses eventual arrival-cohort outcomes for reporting only. Input buckets: "
    "≤512/≤2048/>2048 tokens; output: ≤32/≤128/>128 tokens. TTFT target is an analysis setting, "
    "not a validated patience threshold.</p>",
]
for name in sorted(generated_figures):
    encoded = base64.b64encode((FIGURES / name).read_bytes()).decode("ascii")
    parts.append("<img alt='" + html.escape(name) + "' src='data:image/png;base64," + encoded + "'>")
parts += ["<h2>Reproduction settings</h2><pre>", html.escape(json.dumps(provenance, indent=2)), "</pre></body></html>"]
REPORT = RUN_ROOT / "online_report.html"
REPORT.write_text("\n".join(parts), encoding="utf-8")
ARCHIVE = RUN_ROOT / (run_name + "_online_results.zip")
# No raw/prepared trace; only results, checkpoints, diagnostics and settings.
members = [REPORT, RUN_ROOT / "execution_plan.json", RUN_ROOT / "online_provenance.json", RUN_ROOT / "evaluation_windows.json",
           RUN_ROOT / "attainment_delta.csv", RANGE_CONFIG_PATH, TRAIN_CONFIG_PATH,
           RUN_ROOT / "dataset_summary.csv", RUN_ROOT / "validation_weighted_summary.csv",
           TRAIN_OUTPUT / "pipeline.json", TRAIN_OUTPUT / "validation.csv",
           TRAINING_DIR / "training.csv", TRAINING_DIR / "checkpoints.csv", TRAINING_DIR / "run.json",
           DATA_DIR / "processed/manifest.json", COMMAND_LOG]
members += [TRAINING_DIR / name for name in checkpoint_hashes]
members += sorted(TRAINING_DIR.glob("training_updates_seed*.csv"))
members += [p for p in ONLINE_OUTPUT.rglob("*") if p.is_file()
            and p.suffix in (".json", ".csv", ".npz") and not p.name.endswith(".tmp")]
members += [FIGURES / name for name in sorted(generated_figures)]
with zipfile.ZipFile(ARCHIVE, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in members:
        bundle.write(path, path.relative_to(RUN_ROOT).as_posix())
display(FileLink(str(REPORT)))
display(FileLink(str(ARCHIVE)))
if IS_COLAB:
    from google.colab import files
    files.download(str(ARCHIVE))
print("Outputs remain in:", RUN_ROOT)
